# TP 3 — Partie 5 : surveiller et observer la plateforme

Deux questions différentes, deux familles d'outils, que l'on retrouve dans toutes les équipes data :

| | **Monitoring** (supervision) | **Observabilité des données** |
|---|---|---|
| La question | Les **services** tournent-ils bien ? | Les **données** sont-elles fiables, et d'où viennent-elles ? |
| Ce qu'on regarde | processeur, mémoire, durée et échecs des tâches | *lineage* (qui produit quoi, à partir de quoi), historique des exécutions, schémas |
| Ici | **Prometheus** (collecte des métriques) + **Grafana** (tableaux de bord, alertes) | **OpenLineage** (format standard des événements) + **Marquez** (stockage et interface) |

Tout est déjà configuré : le tableau de bord Grafana, l'alerte, l'envoi du lineage. Dans cette partie, vous **ne programmez rien** : le notebook fait tourner la plateforme, et vous **lisez** les outils, comme une équipe d'exploitation.

> **⚠️ Avant de commencer.** Arrêtez Trino et Superset, puis démarrez les outils de supervision depuis le terminal **Ubuntu** :
> ```bash
> docker compose --profile full stop trino superset
> docker compose --profile observabilite --profile orchestration up -d
> docker compose --profile orchestration restart airflow
> ```
> Le redémarrage d'Airflow est indispensable : Airflow ne cherche le service de métriques qu'à son démarrage. S'il a démarré avant lui, il n'envoie aucune métrique (message `Could not configure StatsClient` dans ses journaux).

In [ ]:
import os
import time

import boto3
import pandas as pd
import requests

AIRFLOW, PROMETHEUS, MARQUEZ = "http://airflow:8080", "http://prometheus:9090", "http://marquez:5000"
TP3 = "/home/etudiant/work/notebooks/mon_travail/tp3"


def promql(requete):
    """Exécute une requête PromQL et renvoie un DataFrame (une ligne par série)."""
    r = requests.get(f"{PROMETHEUS}/api/v1/query", params={"query": requete}, timeout=30).json()
    series = r["data"]["result"]
    if not series:
        print(f"⚠️ Aucune série pour : {requete}\n   (le service mesuré tourne-t-il ? attendez 30 s après son démarrage)")
    return pd.DataFrame([{**s["metric"], "valeur": float(s["value"][1])} for s in series],
                        columns=None if series else ["name", "task_id", "valeur"])


def airflow(methode, chemin, **kw):
    """Appelle l'API REST d'Airflow (en local, pas de mot de passe : un jeton suffit)."""
    jeton = requests.get(f"{AIRFLOW}/auth/token", timeout=30).json()["access_token"]
    r = requests.request(methode, f"{AIRFLOW}/api/v2/{chemin}", timeout=60,
                         headers={"Authorization": f"Bearer {jeton}"}, **kw)
    r.raise_for_status()
    return r.json()


def lancer_dag(dag_id):
    """Active le DAG, le déclenche, puis affiche l'état de ses tâches jusqu'à la fin de l'exécution."""
    for _ in range(20):                                  # le DAG peut mettre 30 s à être lu par Airflow
        try:
            airflow("PATCH", f"dags/{dag_id}", json={"is_paused": False})
            break
        except requests.HTTPError:
            time.sleep(10)
    run = airflow("POST", f"dags/{dag_id}/dagRuns", json={"logical_date": None})["dag_run_id"]
    print(f"{dag_id} : exécution {run} lancée")
    vu = {}
    while True:
        taches = airflow("GET", f"dags/{dag_id}/dagRuns/{run}/taskInstances")["task_instances"]
        for t in taches:
            if t["state"] and vu.get(t["task_id"]) != t["state"]:
                vu[t["task_id"]] = t["state"]
                print(f"  {time.strftime('%H:%M:%S')}  {t['task_id']:22s} {t['state']}")
        etat = airflow("GET", f"dags/{dag_id}/dagRuns/{run}")["state"]
        if etat in ("success", "failed"):
            print(f"{dag_id} : {etat}")
            return etat
        time.sleep(15)


def repond(url):
    try:
        return requests.get(url, timeout=10).ok
    except requests.ConnectionError:
        return False


etat = {"Airflow": repond(f"{AIRFLOW}/api/v2/monitor/health"), "Prometheus": repond(f"{PROMETHEUS}/-/ready"),
        "Marquez": repond(f"{MARQUEZ}/api/v1/namespaces")}
print(" | ".join(f"{nom} : {'OK' if ok else 'injoignable'}" for nom, ok in etat.items()))
if not all(etat.values()):
    print("→ Relancez les commandes ci-dessus dans le terminal Ubuntu, attendez une minute, puis cette cellule.")

## Étape 1 — Faire tourner la plateforme

Les outils de supervision ne montrent que ce qui se passe **pendant qu'ils tournent**. Lors des parties 2 à 4, ils étaient arrêtés : rien n'a été enregistré. La cellule suivante rejoue donc tout le pipeline photo, cette fois sous surveillance :
1. elle vide les tables du TP 3 (les photos des deux lots restent dans la zone d'atterrissage) ;
2. elle déclenche le DAG `catalogue_photos` dans Airflow et suit ses tâches.

Comptez 5 à 8 minutes. Pendant ce temps, ouvrez Grafana (étape 2) et regardez les courbes bouger.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("tp3-reset-supervision").getOrCreate()
for t in ["bronze.photos", "bronze.produits_brut", "silver.photos", "silver.quarantaine_photos",
          "gold.photos_vecteurs", "gold.photos_controle"]:
    spark.sql(f"DROP TABLE IF EXISTS lakehouse.{t} PURGE")
spark.stop()
boto3.client("s3", endpoint_url=os.environ["S3_ENDPOINT"]).delete_object(Bucket="landing", Key="catalogue/_ingerees.txt")
print("tables du TP 3 vidées\n")

lancer_dag("catalogue_photos")

## Étape 2 — Grafana : lire le tableau de bord

Ouvrez http://localhost:3000 (`admin` / `admin`), menu *Dashboards > Lakehouse > Lakehouse — supervision*. En haut à droite, choisissez la période *Last 30 minutes*.

Relevez :
- la mémoire de chaque conteneur pendant l'exécution, et surtout celle de `lakehouse-airflow-1` ;
- la durée de chaque tâche du DAG ;
- le nombre de tâches réussies et en échec.

Les panneaux de Grafana ne sont que des requêtes **PromQL** envoyées à Prometheus. Les mêmes chiffres, lus directement :

In [ ]:
memoire = promql('container_memory_working_set_bytes{name=~"lakehouse-.*"}')
memoire.assign(mo=(memoire.valeur / 1e6).round()).sort_values("mo", ascending=False)[["name", "mo"]]

In [ ]:
durees = promql('airflow_tache_duree_secondes{dag_id="catalogue_photos", quantile="0.5"}')
durees.assign(secondes=durees.valeur.round(1)).sort_values("secondes", ascending=False)[["task_id", "secondes"]]

> **❓ Question 1.** Quel conteneur consomme le plus de mémoire pendant le DAG ? Quelle tâche dure le plus longtemps ? Sur une machine de 8 Go, que pourriez-vous en conclure pour la configuration (`mem_limit`, `spark.driver.memory`) ?

## Étape 3 — Une alerte qui se déclenche

Une alerte est déjà configurée dans Grafana : **Tâche Airflow en échec**, déclenchée dès qu'une tâche échoue. Ouvrez *Alerting > Alert rules* : elle est à l'état *Normal*.

La cellule suivante lance le DAG `panne_simulee`, dont l'unique tâche échoue volontairement (un service distant qui ne répond pas). Puis, dans la minute qui suit, observez l'alerte passer à l'état *Firing*, et le panneau « Tâches en échec » du tableau de bord.

In [ ]:
import shutil

shutil.copy(f"{TP3}/dag_panne_simulee.py", f"{TP3}/dags/panne_simulee.py")
lancer_dag("panne_simulee")

> **❓ Question 2.** Combien de temps s'est-il écoulé entre l'échec de la tâche et le passage de l'alerte à *Firing* ? D'où vient ce délai (pensez à la fréquence de collecte de Prometheus et à celle d'évaluation de l'alerte) ? Pourquoi une alerte vaut-elle mieux qu'un tableau de bord que l'on consulte de temps en temps ?

## Étape 4 — Marquez : le lineage, construit automatiquement

Chaque job Spark lancé par Airflow a envoyé des événements **OpenLineage** à Marquez : ce qu'il a lu, ce qu'il a écrit, quand, avec quel résultat. Personne n'a dessiné le graphe : il est **déduit des exécutions réelles** de l'étape 1.

Ouvrez http://localhost:3001, choisissez l'espace de noms `lakehouse` (en haut à droite), puis cherchez le jeu de données `photos_controle` : le graphe montre toute la chaîne qui le produit. Cliquez sur un job pour voir l'historique de ses exécutions.

In [ ]:
jobs = requests.get(f"{MARQUEZ}/api/v1/namespaces/lakehouse/jobs", params={"limit": 100}, timeout=30).json()["jobs"]
pd.DataFrame([{"job": j["name"], "entrees": [i["name"] for i in j.get("inputs", [])],
               "sorties": [o["name"] for o in j.get("outputs", [])],
               "dernier_etat": (j.get("latestRun") or {}).get("state")} for j in jobs])

> **❓ Question 3.** Vous découvrez que `gold.photos_controle` contient des erreurs. Comment le lineage vous aide-t-il à trouver l'origine du problème ? Et, dans l'autre sens, à savoir qui sera touché si vous modifiez `silver.photos` ?

> **❓ Question 4.** Pour chaque situation, dites si c'est plutôt le monitoring ou l'observabilité des données qui vous alerte : (a) le conteneur Airflow manque de mémoire ; (b) le DAG réussit, mais un vendeur envoie depuis hier des photos en noir et blanc ; (c) `dbt_build` dure deux fois plus longtemps que d'habitude ; (d) une colonne a disparu de `bronze.produits_brut`.

## Pour finir

Remettez en pause le DAG `panne_simulee` dans Airflow, puis, pour libérer la mémoire, arrêtez les outils de supervision depuis le terminal Ubuntu :
```bash
docker compose --profile observabilite stop
```